In [ ]:
import torch

def explore_checkpoint(checkpoint):
    for key in checkpoint.keys():
        print(f"\nKey: {key}")
        value = checkpoint[key]
        
        if isinstance(value, dict):
            print(f"Type: Dictionary with {len(value)} keys")
            for sub_key, sub_value in list(value.items())[:3]:  # Limit to 3 keys
                print(f"  - {sub_key}: {type(sub_value)}")
                if isinstance(sub_value, torch.Tensor):
                    print(f"    Shape: {sub_value.shape}")
        
        elif isinstance(value, list):
            print(f"Type: List with {len(value)} items")
            if value:
                print(f"  First Item Type: {type(value[0])}")
        
        elif isinstance(value, torch.Tensor):
            print(f"Type: Tensor with shape {value.shape}")
        
        else:
            print(f"Type: {type(value)}")
            print(f"  Value: {value}")

# Load the checkpoint
checkpoint = torch.load("/data/geraugi/plural/checkpoints/ofa_base.pt", map_location="cpu")
plural_ckpt = torch.load("/data/geraugi/plural/checkpoints/PLURAL.pt", map_location="cpu")
explore_checkpoint(plural_ckpt)
# Inspect the keys
#print(checkpoint.keys())

In [ ]:
def compare_cfg_structures(cfg1, cfg2, cfg1_name="PLURAL", cfg2_name="OFA", path="root"):
    """
    Compare the structures of two nested configuration dictionaries.
    
    Args:
        cfg1, cfg2 (dict): Configuration dictionaries to compare.
        cfg1_name, cfg2_name (str): Names of the configs (for labeling in output).
        path (str): Current key path (used for nested comparisons).
    """
    cfg1_keys = set(cfg1.keys())
    cfg2_keys = set(cfg2.keys())
    
    # Keys present in cfg1 but missing in cfg2
    for key in cfg1_keys - cfg2_keys:
        print(f"[{path}] Key '{key}' found in {cfg1_name}, but not in {cfg2_name}")
    
    # Keys present in cfg2 but missing in cfg1
    for key in cfg2_keys - cfg1_keys:
        print(f"[{path}] Key '{key}' found in {cfg2_name}, but not in {cfg1_name}")
    
    # Common keys
    for key in cfg1_keys & cfg2_keys:
        value1 = cfg1[key]
        value2 = cfg2[key]
        current_path = f"{path} -> {key}"
        
        # Check type
        if type(value1) != type(value2):
            print(f"[{current_path}] Type mismatch: {cfg1_name}={type(value1)}, {cfg2_name}={type(value2)}")
        
        # If both are dictionaries, recurse
        elif isinstance(value1, dict) and isinstance(value2, dict):
            compare_cfg_structures(value1, value2, cfg1_name, cfg2_name, current_path)
        
        # If both are lists, compare lengths and types
        elif isinstance(value1, list) and isinstance(value2, list):
            if len(value1) != len(value2):
                print(f"[{current_path}] List length mismatch: {cfg1_name}={len(value1)}, {cfg2_name}={len(value2)}")
            else:
                for i, (v1, v2) in enumerate(zip(value1, value2)):
                    if type(v1) != type(v2):
                        print(f"[{current_path}[{i}]] Type mismatch in list: {cfg1_name}={type(v1)}, {cfg2_name}={type(v2)}")
        
        # Otherwise, compare values directly
        else:
            try:
                if value1 != value2:
                    print(f"[{current_path}] Value mismatch: {cfg1_name}={value1}, {cfg2_name}={value2}")
            except Exception as e:
                print(f"Problems comparing values in path {current_path}: {e}")

# Example usage
# Assume plural_cfg and ofa_cfg are loaded dictionaries for PLURAL and OFA configurations.
compare_cfg_structures(plural_ckpt, checkpoint)

In [ ]:
for key in plural_ckpt['cfg'].keys():
    print(f'#############\n {key} \n#############')
    if plural_ckpt['cfg'][key] != None:
        if key != 'model':
            explore_checkpoint(plural_ckpt['cfg'][key])

In [ ]:
explore_checkpoint(checkpoint['model'])

In [ ]:
print(plural_ckpt['last_optimizer_state'].keys())
explore_checkpoint(plural_ckpt['last_optimizer_state'])
print(plural_ckpt['last_optimizer_state']['state'])
print(plural_ckpt['last_optimizer_state']['param_groups'])
print(plural_ckpt['last_optimizer_state']['loss_scale'])

In [ ]:
checkpoint=plural_ckpt
print(checkpoint['extra_state'].keys())
explore_checkpoint(checkpoint['extra_state'])
print("### TRAIN ITERATOR ###")
explore_checkpoint(checkpoint['extra_state']['train_iterator'])
print("### METRICS ###")
explore_checkpoint(checkpoint['extra_state']['metrics'])
for key in checkpoint['extra_state']['metrics']:
    print(f'\n### {key} ###\n')
    for element in checkpoint['extra_state']['metrics'][key]:
        print(element)
print(checkpoint['extra_state']['previous_training_time'])

In [ ]:
for i in range(10):
    print(checkpoint['optimizer_history'][i])
#explore_checkpoint(checkpoint['optimizer_history'])

In [ ]:
print(checkpoint['cfg'].keys())
explore_checkpoint(checkpoint['cfg'])

In [ ]:
for key in checkpoint['cfg'].keys():
    print(f'#############\n {key} \n#############')
    if checkpoint['cfg'][key] != None:
        if key != 'model':
            explore_checkpoint(checkpoint['cfg'][key])

In [ ]:
print(checkpoint['cfg']['optimization']['stop_min_lr'])
print(checkpoint['cfg']['optimizer']['lr'])
print(checkpoint['cfg']['lr_scheduler']['lr'])
print(checkpoint['cfg']['optimization']['update_freq'])
print(checkpoint['cfg']['distributed_training']['distributed_world_size'])
print(checkpoint['cfg']['dataset']['batch_size'])